# What makes the same person look different? (MEVID-SC)

**Question:** apart from camera shift, which factor causes CSCI (and the OSNet baseline) to split one person into several IDs across cameras?

**How:** every clip is measured automatically, with no manual labels:

| Factor | How it is measured |
|---|---|
| Viewpoint (front, back, side) | Pose model: the face is visible and the shoulder order agrees; narrow shoulders mean side |
| Distance | Person height in pixels |
| Size mismatch | Ratio of the two clips' heights |
| Occlusion | Share of body keypoints not visible, when the crop does not cut the body |
| Crop cuts the body | Head or feet missing at the crop edge |
| Crop alignment | How much of the crop the detected person fills |
| Another person in crop | A second person detected inside the crop |
| Blur | Sharpness (Laplacian variance) |
| Lighting change | Brightness difference between the two clips |
| Clip length | Number of frames |
| Camera pair | Which two cameras (camera shift), for comparison |

A **split** is a pair of clips of the same person, on different cameras, that ended up with different IDs at 90% pair precision.

**Ranking:** all factors go into one model together, with a fixed effect per person, so only clips of the same person are compared. Each person is weighted equally. Both numbers are corrected for chance, and 95% intervals come from bootstrapping people:
- **alone:** what the factor explains by itself;
- **unique:** what it adds after all the other factors.

Settings are frozen in cell 1 before any result is seen.

**Runtime:** about 15 to 20 minutes on a GPU runtime. Run cells 1 to 7 in order, and look at the images from cell 4 before trusting cell 6.

**Validation:** the analysis was tested on synthetic data with a planted cause (viewpoint first, distance second), and it recovered both in the right order.

In [ ]:
# 1. Setup: mount Drive, find MEVID-SC, its annotations and the saved
#    fingerprints (CSCI and the OSNet baseline). All settings are fixed here,
#    before any result is seen.
!pip -q install ultralytics
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
CS = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
                           'test_templates_csci_sc.npz',
                           f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz']))
TR = os.path.dirname(find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
                           'test_templates_tracker_sc.npz',
                           f'{SD}/**/Tracker-SameClothes/test_templates_tracker_sc.npz']))
TAR = find([f'{SC_DIR}/mevid-sc-bbox-test.tar', f'{SC_DIR}/*test*.tar'])
OUT = os.path.join(os.path.dirname(CS), 'LookFactors-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True),
                           1e-8)).astype(np.float32)


A = f'{SC_DIR}/annotations'
names = [l.strip() for l in open(f'{A}/test_name.txt') if l.strip()]
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
pid, cam = info[:, 2], info[:, 4]
te = np.load(f'{CS}/test_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
assert (te['pid'] == pid).all() and (te['cam'] == cam).all(), 'CSCI order'
assert (tte['pid'] == pid).all() and (tte['cam'] == cam).all(), 'tracker order'
FEATS = {'CSCI': unit(te['templates']),
         'OSNet baseline (OSNet)': unit(tte['zs'])}

SETTINGS = {'frames_per_clip': 8, 'pose_model': 'yolo11m-pose.pt',
            'imgsz': 320, 'keypoint_conf': 0.5,
            'other_person_conf': 0.5, 'other_person_min_area': 0.15,
            'side_ratio': 0.35, 'edge': 0.03,
            'precision_target': 0.90, 'n_boot': 200, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print(f'{len(info)} clips, {len(np.unique(pid))} people, '
      f'{len(np.unique(cam))} cameras | results -> {OUT}')


In [ ]:
# 2. Copy a few evenly spaced frames of every clip out of the tar
#    (8 per clip, about 11,000 images). Read once; reruns skip this.
LOCAL = '/content/mevid_sc_frames'
os.makedirs(LOCAL, exist_ok=True)
want = {}
for k, (s, e) in enumerate(info[:, :2]):
    n = min(e - s + 1, SETTINGS['frames_per_clip'])
    for i in np.unique(np.linspace(s, e, n).round().astype(int)):
        want[names[i]] = k
have = set(os.listdir(LOCAL))
if len(have & set(want)) < len(want):
    import tarfile
    from tqdm.auto import tqdm
    t0 = time.time()
    bar = tqdm(total=len(want) - len(have & set(want)), desc='frames')
    with tarfile.open(TAR, 'r|') as t:
        for m in t:
            b = os.path.basename(m.name)
            if m.isfile() and b in want and b not in have:
                with open(f'{LOCAL}/{b}', 'wb') as f:
                    f.write(t.extractfile(m).read())
                have.add(b); bar.update(1)
    bar.close()
    print(f'copied in {time.time() - t0:.0f} s')
missing = [n for n in want if n not in have]
assert not missing, f'{len(missing)} frames not found, e.g. {missing[:3]}'
print(f'{len(want)} frames ready from {len(info)} clips')


In [ ]:
# 3. Measure every frame automatically (no manual labels):
#    size in pixels (distance), sharpness (blur), brightness (lighting),
#    and with a pose model: viewpoint, visible body parts (occlusion),
#    body cut by the crop edge, crop alignment, another person in the crop.
import cv2
from ultralytics import YOLO
from tqdm.auto import tqdm
st = SETTINGS
pose = YOLO(st['pose_model'])
KC, EDGE = st['keypoint_conf'], st['edge']


def viewpoint(kp, v):
    """front / back / side from COCO keypoints. Two cues must agree:
    face visible (nose + an eye) and shoulder order (a person facing the
    camera has their left shoulder on the image right). Narrow shoulders
    relative to the torso = side. Disagreement = unknown."""
    if not (v[5] and v[6]):
        return 'unknown'
    sw = abs(kp[5, 0] - kp[6, 0])
    if v[11] and v[12]:
        torso = abs((kp[11, 1] + kp[12, 1]) / 2 - (kp[5, 1] + kp[6, 1]) / 2)
        if torso > 0 and sw < st['side_ratio'] * torso:
            return 'side'
    face = v[0] and (v[1] or v[2])
    order = kp[5, 0] > kp[6, 0]
    if face and order:
        return 'front'
    if not face and not order:
        return 'back'
    return 'unknown'


files = sorted(want)
rows = []
for i in tqdm(range(0, len(files), 64), desc='measuring'):
    batch = files[i:i + 64]
    imgs = [cv2.imread(f'{LOCAL}/{f}') for f in batch]
    res = pose.predict(imgs, imgsz=st['imgsz'], conf=0.25, verbose=False)
    for f, im, r in zip(batch, imgs, res):
        H, W = im.shape[:2]
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        g256 = cv2.resize(g, (max(1, int(round(W * 256 / H))), 256))
        row = dict(name=f, clip=want[f], height=H,
                   sharp=float(np.log1p(cv2.Laplacian(g256, cv2.CV_64F).var())),
                   bright=float(g.mean()), found=0, fill=0.0, others=0,
                   visible=0.0, cut=0, view='unknown')
        if len(r.boxes):
            xy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
            area = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1]) / (H * W)
            m = int(np.argmax(area * cf))          # the clip's person
            kp = r.keypoints.data.cpu().numpy()[m]
            v = kp[:, 2] >= KC
            oth = (cf >= st['other_person_conf']) & \
                  (area >= st['other_person_min_area'])
            oth[m] = False
            head_cut = (not v[:5].any()) and xy[m, 1] <= EDGE * H
            feet_cut = (not v[15:17].any()) and xy[m, 3] >= (1 - EDGE) * H
            row.update(found=1, fill=float(min(area[m], 1.0)),
                       others=int(oth.sum()), visible=float(v.mean()),
                       cut=int(head_cut or feet_cut), view=viewpoint(kp, v))
        rows.append(row)
FR = pd.DataFrame(rows)
FR.to_csv(f'{OUT}/frame_measurements.csv', index=False)


def majority(s):
    s = s[s != 'unknown']
    return s.value_counts().index[0] if len(s) else 'unknown'


g = FR.groupby('clip')
CL = pd.DataFrame({'height': g.height.median(), 'sharp': g.sharp.median(),
                   'bright': g.bright.median(), 'fill': g.fill.median(),
                   'visible': g.visible.median(), 'cut': g.cut.mean(),
                   'occluded': g.apply(lambda d: ((d.visible < 0.6)
                                                  & (d.cut == 0)).mean()),
                   'crowd': g.others.apply(lambda s: (s > 0).mean()),
                   'found': g.found.mean(), 'view': g.view.agg(majority)}
                  ).reindex(range(len(info)))
CL['frames'] = info[:, 1] - info[:, 0] + 1
CL['pid'], CL['cam'] = pid, cam
CL.to_csv(f'{OUT}/clip_measurements.csv')
print('Person found by the pose model in', f'{100 * FR.found.mean():.1f}%',
      'of frames')
print('\nClip viewpoints:'); display(CL.view.value_counts().to_frame('clips'))
display(CL[['height', 'sharp', 'bright', 'fill', 'visible', 'cut',
            'occluded', 'crowd', 'frames']].describe().round(2))


In [ ]:
# 4. Visual check of the automatic measurements: look at these images.
#    If viewpoints or cuts look wrong, say so before trusting cell 6.
import matplotlib.pyplot as plt
rng = np.random.default_rng(st['seed'])
mid = FR.groupby('clip').name.apply(lambda s: s.iloc[len(s) // 2])


def grid(groups, title, path, n=8):
    fig, ax = plt.subplots(len(groups), n, figsize=(n * 1.6, len(groups) * 3.2))
    ax = np.atleast_2d(ax)
    for r, (lab, clips) in enumerate(groups.items()):
        pick = rng.choice(clips, min(n, len(clips)), replace=False) \
            if len(clips) else []
        for c in range(n):
            a = ax[r, c]; a.axis('off')
            if c < len(pick):
                im = cv2.imread(f'{LOCAL}/{mid[pick[c]]}')[:, :, ::-1]
                a.imshow(im)
            if c == 0:
                a.set_title(lab, loc='left', fontsize=11)
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(path, dpi=110); plt.show()


grid({v: CL.index[CL.view == v].to_numpy()
      for v in ['front', 'back', 'side', 'unknown']},
     'Viewpoint, as measured (random clips)', f'{OUT}/check_viewpoint.png')
q = CL.height.quantile([1 / 3, 2 / 3]).to_numpy()
grid({'far (small)': CL.index[CL.height < q[0]].to_numpy(),
      'near (large)': CL.index[CL.height >= q[1]].to_numpy(),
      'body cut by crop': CL.index[CL.cut > 0.5].to_numpy(),
      'occluded': CL.index[CL.occluded > 0.5].to_numpy(),
      'another person in crop': CL.index[CL.crowd > 0.3].to_numpy()},
     'Distance, cuts, occlusion, crowding (random clips)',
     f'{OUT}/check_factors.png')


In [ ]:
# 5. Where are people split? Consistent ID by clustering (average
#    linkage) at the cut that gives 90% pair precision (diagnostic, uses
#    labels only to place the operating point). A "split" is a pair of
#    clips of the SAME person on DIFFERENT cameras that got different IDs.
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


def pair_precision(lab, p):
    M = pd.crosstab(p, lab).to_numpy().astype(float)
    nc = M.sum(0)
    same = (nc * (nc - 1)).sum()
    return (M * (M - 1)).sum() / same if same else 1.0   # nothing merged yet


def ids_at_precision(X, p, target):
    D = np.clip(1 - X @ X.T, 0, None); D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    L = linkage(squareform(D, checks=False), 'average')
    best = fcluster(L, 1e-9, 'distance')
    for t in np.linspace(0.01, 1.5, 150):
        lab = fcluster(L, t, 'distance')
        if pair_precision(lab, p) >= target:
            best = lab
        else:
            break
    return best


iu, ju = np.triu_indices(len(pid), 1)
keep = (pid[iu] == pid[ju]) & (cam[iu] != cam[ju])
I, J = iu[keep], ju[keep]
P = pd.DataFrame({'i': I, 'j': J, 'person': pid[I]})
for bb, X in FEATS.items():
    lab = ids_at_precision(X, pid, st['precision_target'])
    P[f'sim {bb}'] = (X[I] * X[J]).sum(1)
    P[f'split {bb}'] = (lab[I] != lab[J]).astype(float)
    print(f'{bb}: {pd.Series(lab).groupby(pid).nunique().mean():.2f} IDs per '
          f'person, {100 * P[f"split {bb}"].mean():.1f}% of same-person '
          'cross-camera pairs split')
print(f'{len(P):,} same-person cross-camera pairs, {P.person.nunique()} people')


In [ ]:
# 6. Which factor explains the splits most?
#    Each pair gets one level per factor (thresholds fixed in advance or
#    by tertiles of the clips, never tuned on splits). Then:
#    (a) plain table: split rate at each level of each factor;
#    (b) ranking: a model with ALL factors together and a fixed effect per
#        person (so it compares clips of the same person), each person
#        weighted equally. "Alone" = variance in splits the factor explains
#        by itself; "unique" = what it adds after all other factors.
#        Both are corrected for chance (shuffled-factor baseline).
#        95% intervals: 200 bootstrap resamples of people.
C = CL


def tert(x, hi_good=True, labels=('low', 'mid', 'high')):
    q = np.quantile(C[x].dropna(), [1 / 3, 2 / 3])
    return q


def lev3(v, q, names):
    return np.where(v < q[0], names[0], np.where(v < q[1], names[1], names[2]))


vi, vj = C.view.to_numpy()[P.i], C.view.to_numpy()[P.j]
both = np.where((vi == 'unknown') | (vj == 'unknown'), 'view unknown',
       np.where(vi == vj, 'same view',
       np.where(((vi == 'front') & (vj == 'back')) |
                ((vi == 'back') & (vj == 'front')), 'front vs back',
                'side vs other')))
h = C.height.to_numpy()
hmin, hrat = np.minimum(h[P.i], h[P.j]), np.maximum(h[P.i], h[P.j]) / \
    np.minimum(h[P.i], h[P.j])
F = pd.DataFrame(index=P.index)
F['Viewpoint'] = both
F['Distance (person size)'] = lev3(hmin, tert('height'), ['far', 'middle', 'near'])
F['Size mismatch'] = lev3(hrat, [1.5, 2.5], ['similar', 'x1.5-2.5', '>x2.5'])
occ = np.maximum(C.occluded.to_numpy()[P.i], C.occluded.to_numpy()[P.j])
F['Occlusion'] = lev3(occ, [0.25, 0.5], ['none', 'some', 'heavy'])
F['Crop cuts the body'] = np.where(np.maximum(C.cut.to_numpy()[P.i],
                                              C.cut.to_numpy()[P.j]) > 0.5,
                                   'cut', 'whole')
fl = np.minimum(C.fill.to_numpy()[P.i], C.fill.to_numpy()[P.j])
F['Crop alignment'] = lev3(fl, tert('fill'), ['poor', 'fair', 'good'])
F['Another person in crop'] = np.where(np.maximum(C.crowd.to_numpy()[P.i],
                                                  C.crowd.to_numpy()[P.j]) > 0.3,
                                       'yes', 'no')
sh = np.minimum(C.sharp.to_numpy()[P.i], C.sharp.to_numpy()[P.j])
F['Blur'] = lev3(sh, tert('sharp'), ['blurry', 'fair', 'sharp'])
bd = np.abs(C.bright.to_numpy()[P.i] - C.bright.to_numpy()[P.j])
qb = np.quantile(np.abs(C.bright.to_numpy()[:, None] - C.bright.to_numpy()[None]),
                 [1 / 3, 2 / 3])
F['Lighting change'] = lev3(bd, qb, ['small', 'medium', 'large'])
fr = np.minimum(C.frames.to_numpy()[P.i], C.frames.to_numpy()[P.j])
F['Clip length'] = lev3(fr, tert('frames'), ['short', 'medium', 'long'])
ci, cj = cam[P.i], cam[P.j]
F['Camera pair (camera shift)'] = [f'{min(a, b)}-{max(a, b)}'
                                   for a, b in zip(ci, cj)]
EASY = {'Viewpoint': 'same view', 'Distance (person size)': 'near',
        'Size mismatch': 'similar', 'Occlusion': 'none',
        'Crop cuts the body': 'whole', 'Crop alignment': 'good',
        'Another person in crop': 'no', 'Blur': 'sharp',
        'Lighting change': 'small', 'Clip length': 'long'}
FACTORS = list(F.columns)
gcode = pd.factorize(P.person)[0]
W = 1.0 / np.bincount(gcode)[gcode]                # each person counts once
W = W / W.mean()


def dummies(f):
    d = pd.get_dummies(F[f]).astype(float)
    ref = EASY.get(f, d.sum().idxmax())
    return d.drop(columns=[ref]).to_numpy() if ref in d else d.to_numpy()[:, 1:]


DUM = {f: dummies(f) for f in FACTORS}


def demean(M, g):
    s = np.zeros((g.max() + 1, M.shape[1])); np.add.at(s, g, M)
    return M - (s / np.bincount(g)[:, None])[g]


def r2(y, X, g, w):
    if X.shape[1] == 0:
        return 0.0
    yd, Xd, sw = demean(y[:, None], g)[:, 0], demean(X, g), np.sqrt(w)
    sst = ((yd * sw) ** 2).sum()
    if sst <= 0:
        return 0.0
    b = np.linalg.lstsq(Xd * sw[:, None], yd * sw, rcond=None)[0]
    return 1 - (((yd - Xd @ b) * sw) ** 2).sum() / sst


def importance(y, rows, g, w, dum=None):
    dum = dum or DUM
    full = np.hstack([dum[f][rows] for f in FACTORS])
    R = r2(y[rows], full, g, w[rows])
    out = {'_all': R}
    for f in FACTORS:
        rest = np.hstack([dum[k][rows] for k in FACTORS if k != f])
        out[f + '|alone'] = r2(y[rows], dum[f][rows], g, w[rows])
        out[f + '|unique'] = R - r2(y[rows], rest, g, w[rows])
    return out


def chance_level(y, n_perm=10):
    # A factor with many levels (camera pair has ~50) explains some
    # variation by chance alone. Estimate that by shuffling each factor's
    # levels WITHIN each person, then subtract it from every estimate.
    rs = np.random.default_rng(st['seed'] + 1)
    allr = np.arange(len(P))
    null = {}
    for f in FACTORS:
        a = []
        for _ in range(n_perm):
            perm = allr.copy()
            for rows in by_person:
                perm[rows] = rs.permutation(rows)
            d = dict(DUM); d[f] = DUM[f][perm]
            o = importance(y, allr, gcode, W, d)
            a.append((o[f + '|alone'], o[f + '|unique']))
        null[f + '|alone'], null[f + '|unique'] = np.mean(a, 0)
    return null


def attributable(y, rows, w):
    out = {}
    for f, ref in EASY.items():
        lv = F[f].to_numpy()[rows]; yy, ww = y[rows], w[rows]
        r = lv == ref
        if r.sum() == 0 or (yy * ww).sum() == 0:
            out[f] = np.nan; continue
        base = (yy[r] * ww[r]).sum() / ww[r].sum()
        out[f] = max(0.0, 1 - base * ww.sum() / (yy * ww).sum())
    return out


people = np.unique(gcode)
by_person = [np.flatnonzero(gcode == k) for k in people]
rng = np.random.default_rng(st['seed'])
RANK, LEVELS = {}, []
for bb in FEATS:
    y = P[f'split {bb}'].to_numpy()
    allr = np.arange(len(P))
    null = chance_level(y)
    point = importance(y, allr, gcode, W)
    point = {k: v - null.get(k, 0.0) for k, v in point.items()}
    att = attributable(y, allr, W)
    boots, batt = [], []
    for b in tqdm(range(st['n_boot']), desc=bb):
        pick = rng.choice(len(people), len(people), replace=True)
        rows = np.concatenate([by_person[k] for k in pick])
        g = np.concatenate([np.full(len(by_person[k]), n)
                            for n, k in enumerate(pick)])
        o = importance(y, rows, g, W)
        boots.append({k: v - null.get(k, 0.0) for k, v in o.items()})
        batt.append(attributable(y, rows, W))
    B = pd.DataFrame(boots); BA = pd.DataFrame(batt)
    tab = []
    for f in FACTORS:
        row = {'factor': f}
        for kind in ['alone', 'unique']:
            lo, hi = np.percentile(B[f'{f}|{kind}'], [2.5, 97.5])
            row[f'{kind} % (95% CI)'] = (f'{100 * point[f"{f}|{kind}"]:.1f} '
                                         f'[{100 * lo:.1f}, {100 * hi:.1f}]')
            row[f'_{kind}'] = point[f'{f}|{kind}']
        if f in att and not np.isnan(att[f]):
            lo, hi = np.nanpercentile(BA[f], [2.5, 97.5])
            row['splits removed if easy % (95% CI)'] = (
                f'{100 * att[f]:.0f} [{100 * lo:.0f}, {100 * hi:.0f}]')
        else:
            row['splits removed if easy % (95% CI)'] = 'n/a'
        tab.append(row)
    T = pd.DataFrame(tab).sort_values('_unique', ascending=False)
    pos = T._unique.clip(lower=0)
    T.insert(1, 'share of explained splits %',
             (100 * pos / max(pos.sum(), 1e-12)).round(0))
    T.insert(0, 'rank', range(1, len(T) + 1))
    RANK[bb] = T
    print(f'\n=== {bb}: what explains the ID splits '
          f'(all factors together explain {100 * point["_all"]:.1f}% '
          'of the variation in splits within a person) ===')
    display(T.drop(columns=['_alone', '_unique']).set_index('rank'))
    T.to_csv(f'{OUT}/factor_ranking_{bb.split()[0]}.csv', index=False)
    for f in FACTORS:
        if f.startswith('Camera pair'):
            continue
        d = pd.DataFrame({'level': F[f], 'split': y, 'w': W})
        for lv, dd in d.groupby('level'):
            LEVELS.append({'backbone': bb, 'factor': f, 'level': lv,
                           'pairs': len(dd), 'share of pairs %':
                           round(100 * len(dd) / len(d), 1),
                           'split rate %': round(100 * np.average(
                               dd.split, weights=dd.w), 1)})
LV = pd.DataFrame(LEVELS)
LV.to_csv(f'{OUT}/split_rate_by_level.csv', index=False)
print('\nSplit rate at each level (each person weighted equally):')
display(LV.pivot_table(index=['factor', 'level'], columns='backbone',
                       values='split rate %'))


In [ ]:
# 7. The most-split people: one frame per ID they were split into, with
#    the measured factors under each frame. Shows the causes directly.
X = FEATS['CSCI']
lab = ids_at_precision(X, pid, st['precision_target'])
nid = pd.Series(lab).groupby(pid).nunique().sort_values(ascending=False)
top = nid.index[:6]
fig, ax = plt.subplots(len(top), 6, figsize=(13, len(top) * 3.4))
for r, p in enumerate(top):
    ids = pd.Series(lab[pid == p]).value_counts().index[:6]
    for c in range(6):
        a = ax[r, c]; a.axis('off')
        if c < len(ids):
            k = np.flatnonzero((pid == p) & (lab == ids[c]))[0]
            a.imshow(cv2.imread(f'{LOCAL}/{mid[k]}')[:, :, ::-1])
            a.set_title(f'cam {cam[k]} | {CL.view[k]}\n'
                        f'{CL.height[k]:.0f}px vis {CL.visible[k]:.2f}',
                        fontsize=8)
        if c == 0:
            a.text(-0.1, 0.5, f'person {p}\n{nid[p]} IDs', transform=a.transAxes,
                   ha='right', va='center', fontsize=9)
fig.suptitle('CSCI: the most-split people, one frame per ID')
fig.tight_layout(); fig.savefig(f'{OUT}/most_split_people.png', dpi=110)
plt.show()
import shutil
shutil.make_archive('/content/LookFactors_results', 'zip', OUT)
print('Saved to', OUT, '\nZip: /content/LookFactors_results.zip')
